In [1]:
pip install mesa 

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install rdflib

Note: you may need to restart the kernel to use updated packages.


In [1]:
from mesa import Agent, Model
from mesa.time import RandomActivation
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, OWL, XSD

In [7]:
# Define the namespace based on your ontology's IRI
NS = Namespace("https://gist.github.com/ad25be8c0ab03460c8dc.git")

In [9]:
class BookingRequestAgent(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.requested_seat = None

    def step(self):
        # For demo purposes, we'll request seats in order
        seat_number = (self.unique_id - 1) % 10 + 1
        self.requested_seat = f"Seat{seat_number}"
        print(f"Agent {self.unique_id}: Requesting booking for {self.requested_seat}")
        self.model.booking_agent.process_request(self)

In [13]:
class BookingAgent(Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.graph = Graph()
        self.graph.parse("books.owl", format="xml")  # Load the ontology

    def process_request(self, request_agent):
        seat_uri = NS[request_agent.requested_seat]
        
        # Check if the seat is available by traversing the graph
        for obj in self.graph.objects(seat_uri, NS.isAvailable):
            if obj == Literal(True, datatype=XSD.boolean):
                print(f"Agent {self.unique_id}: Booking confirmed for {request_agent.requested_seat}")
                # Mark the seat as unavailable after booking - updating rdf file
                self.graph.set((seat_uri, NS.isAvailable, Literal(False, datatype=XSD.boolean)))
                break
        else:
            print(f"Agent {self.unique_id}: {request_agent.requested_seat} is not available")

In [11]:
class BookingModel(Model):
    def __init__(self):
        self.schedule = RandomActivation(self)
        
        self.booking_agent = BookingAgent(1, self)
        self.schedule.add(self.booking_agent)

        # Initialize agents with unique IDs
        self.request_agents = [BookingRequestAgent(i + 2, self) for i in range(10)]
        for agent in self.request_agents:
            self.schedule.add(agent)

    def step(self):
        self.schedule.step()

In [14]:
# Run the model
if __name__ == '__main__':
    model = BookingModel()
    for _ in range(10):
        model.step()

Agent 10: Requesting booking for Seat10
Agent 1: Seat10 is not available
Agent 7: Requesting booking for Seat7
Agent 1: Seat7 is not available
Agent 11: Requesting booking for Seat1
Agent 1: Seat1 is not available
Agent 2: Requesting booking for Seat2
Agent 1: Seat2 is not available
Agent 5: Requesting booking for Seat5
Agent 1: Seat5 is not available
Agent 6: Requesting booking for Seat6
Agent 1: Seat6 is not available
Agent 9: Requesting booking for Seat9
Agent 1: Seat9 is not available
Agent 4: Requesting booking for Seat4
Agent 1: Seat4 is not available
Agent 8: Requesting booking for Seat8
Agent 1: Seat8 is not available
Agent 3: Requesting booking for Seat3
Agent 1: Seat3 is not available
Agent 2: Requesting booking for Seat2
Agent 1: Seat2 is not available
Agent 9: Requesting booking for Seat9
Agent 1: Seat9 is not available
Agent 11: Requesting booking for Seat1
Agent 1: Seat1 is not available
Agent 8: Requesting booking for Seat8
Agent 1: Seat8 is not available
Agent 6: Reques

C:\Users\marap\AppData\Local\Temp\ipykernel_12180\467096642.py:3: FutureWarning: The Mesa Model class was not initialized. In the future, you need to explicitly initialize the Model by calling super().__init__() on initialization.
  super().__init__(unique_id, model)
